<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/01-glowtts-from-scratch/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Coqui TTS has dependency-injection style api for working with its function and classes.

I am using GlowTTS and LJSpeech for trainig single speaker text-to-speech model.

In [ ]:
!nvidia-smi

In [ ]:
!pip install coqui-tts

In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.57.5

In [ ]:
from TTS.api import TTS

print("TTS OK")

In [ ]:
!pip freeze > requirements.txt

In [ ]:
# download LJSpeech
from TTS.utils.downloaders import download_ljspeech
dataset_path = download_ljspeech("./data")
print(dataset_path)

In [ ]:
# verify dataset
import os
import soundfile as sf

path = "./data/LJSpeech-1.1/wavs"

files = os.listdir(path)

bad = 0

for f in files[:100]:
    try:
        audio, sr = sf.read(os.path.join(path, f))
        if sr != 22050:
            bad += 1
    except:
        bad += 1

print("bad:", bad)
print("files:", len(files))

In [ ]:
# read transcripts
import pandas as pd

path = "./data/LJSpeech-1.1"
metadata = pd.read_csv(
    os.path.join(path, "metadata.csv"),
    sep="|",
    header=None,
)

metadata.head()

In [ ]:
# create dataset config
from TTS.tts.configs.shared_configs import BaseDatasetConfig

dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    path="./data/LJSpeech-1.1/",
    meta_file_train="metadata.csv"
)

In [ ]:
# GlowTTS model config
from TTS.tts.configs.glow_tts_config import GlowTTSConfig

config = GlowTTSConfig(
    batch_size = 16,
    eval_batch_size = 8,
    num_loader_workers = 0,
    num_eval_loader_workers = 0,
    run_eval = True,
    epochs = 100,
    text_cleaner = "english_cleaners",
    use_phonemes = False,
    print_step = 25,
    mixed_precision = True,
    output_path = "./checkpoints",
    datasets = [dataset_config],
)

print("GlowTTS config ready")

In [ ]:
start_epoch = load_checkpoint_if_available(model)
print(f"Resuming from epoch {start_epoch}")

In [ ]:
# prepare data for model training flow
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.tts.datasets import load_tts_samples

# convert wav files into mel-spectrograms
ap = AudioProcessor.init_from_config(config)

# convert text into token IDs
tokenizer, config = TTSTokenizer.init_from_config(config)

# load dataset
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_size=config.eval_split_size,
    eval_split_max_size=config.eval_split_max_size
)

print("Train: ", len(train_samples))
print("Eval: ", len(eval_samples))

In [ ]:
# GlowTTS model initialization
from TTS.tts.models.glow_tts import GlowTTS

model = GlowTTS(
    config,
    ap,
    tokenizer,
    speaker_manager=None
)

print("GlowTTS model ready")

In [ ]:
# trainer setup
from trainer import Trainer, TrainerArgs

trainer = Trainer(
    TrainerArgs(),
    config,
    config.output_path,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples
)

print("Trainer ready")

In [ ]:
# training
trainer.fit()

In [ ]:
# tensorboard monitoring
%load_ext tensorboard

%tensorboard --logdir ./checkpoints